<a href="https://colab.research.google.com/github/mf2056/F20AA_CW2/blob/main/Step6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd

# Load the model-ready CSV
train = pd.read_csv("train_processed.csv")

# Quick check to see if the ratings are 1-5 or 0-4
print(train['rating'].unique())

[2 1 4 5 3]


In [5]:
print(train.columns)

Index(['text', 'rating', 'tokens', 'stemmed', 'lemmatized', 'text_stem',
       'text_lemma', 'review_length'],
      dtype='object')


### 1. Data Sampling and Subset Preparation

In [6]:
# Isolate 1-star and 5-star reviews
low_count = min(len(train[train['rating'] == 1]), 5000)
high_count = min(len(train[train['rating'] == 5]), 5000)

df_low = train[train['rating'] == 1].sample(n=low_count, random_state=42)
df_high = train[train['rating'] == 5].sample(n=high_count, random_state=42)

print(f"Sampling {low_count} 1-star reviews and {high_count} 5-star reviews.")

Sampling 5000 1-star reviews and 5000 5-star reviews.


### 2. Text Vectorization (Bag-of-Words Representation)

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# Use CountVectorizer with lemmatized text
vectorizer = CountVectorizer(max_df=0.9, min_df=5, stop_words='english')

# Transform Low Ratings
dtm_low = vectorizer.fit_transform(df_low['text_lemma'].astype(str).fillna(''))
vocab_low = vectorizer.get_feature_names_out()

# Transform High Ratings
dtm_high = vectorizer.fit_transform(df_high['text_lemma'].astype(str).fillna(''))
vocab_high = vectorizer.get_feature_names_out()

### 3. LDA Model Training and Topic Extraction

In [8]:
from sklearn.decomposition import LatentDirichletAllocation

# Fit LDA Models for both categories (15 Topics each)
lda_low = LatentDirichletAllocation(n_components=15, random_state=42, n_jobs=-1)
lda_low.fit(dtm_low)

lda_high = LatentDirichletAllocation(n_components=15, random_state=42, n_jobs=-1)
lda_high.fit(dtm_high)

# Function to print the extracted topics
def print_top_words(model, feature_names, n_top_words, title):
    print(f"\n=== {title} ===")
    for topic_idx, topic in enumerate(model.components_):
        message = f"Topic #{topic_idx + 1}: "
        message += " | ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)
        print("-" * 10)

print_top_words(lda_low, vocab_low, 10, "1-STAR TOPICS")
print_top_words(lda_high, vocab_high, 10, "5-STAR TOPICS")


=== 1-STAR TOPICS ===
Topic #1: place | apartment | rent | management | dont | property | year | month | lease | stay
----------
Topic #2: asked | time | said | told | didnt | hair | want | like | store | make
----------
Topic #3: money | company | dont | owner | know | business | job | work | time | day
----------
Topic #4: nail | like | got | place | didnt | time | dont | cut | price | went
----------
Topic #5: store | return | gas | order | mask | hair | review | star | know | item
----------
Topic #6: doctor | appointment | patient | office | dr | time | staff | insurance | told | care
----------
Topic #7: hour | open | people | time | wait | waiting | package | pm | door | dont
----------
Topic #8: company | time | year | month | email | house | told | business | bank | issue
----------
Topic #9: car | price | vehicle | shop | guy | took | repair | place | said | company
----------
Topic #10: hospital | test | help | told | time | called | work | said | result | dont
----------
T

### Step 6: Analysis and Observations

**1. Categorization of Topics:**
* **Low Rating (1-Star):** Topics center on functional failures and friction. Major themes include **Property & Rent Management** (Topic #1), **Medical & Hospital Friction** (Topics #6, #10), and **Staff Conduct/Rudeness** (Topic #14).
* **Financial/Admin Issues:** Topics #3, #8, and #15 highlight disputes over money, banking, and credit cards.
* **High Rating (5-Star):** Topics center on experiential satisfaction. Major themes include **Professional Service & Competence** (Topics #4, #7, #8) and **Interpersonal Excellence** (Topics #3, #15).

**2. Key Inferences:**
* **Specificity vs. Generality:** 1-star reviews are highly specific and "event-driven" (e.g., issues with lease agreements, car repairs, or pharmacy prescriptions). 5-star reviews use more general, superlative language (e.g., "amazing," "best," "highly recommend").
* **The Role of Time:** In negative reviews (Topic #7), "hour" and "wait" represent a resource lost. In positive reviews, "time" is associated with professional consistency and a "job well done."
* **Sentiment Drivers:** This LDA analysis demonstrates that 1-star ratings are primarily driven by objective process failures (billing, long waits, property issues), whereas 5-star ratings are driven by subjective emotional satisfaction and staff helpfulness.